# T18 / E09 — Tinh chỉnh ba bộ mã hóa làm mốc so sánh

Đây là **mốc so sánh nghiêm túc nhất** của đề tài. Khác baseline tầm thường ở E01, ba mô hình
này thật sự đọc văn bản; khác LLM giám khảo ở E10, chúng chạy cục bộ không tốn API. Nếu phương
pháp chú ý nội tại không vượt được chúng thì lập luận về chi phí ở câu hỏi CH2 không đứng vững.

**Notebook settings:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens`
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

## Ước tính thời gian

Đo thật ở lượt chạy ngày 27/08/2026, không phải ước lượng:

| Mô hình | Độ dài | Batch | lr | 3 seed |
|---|---|---|---|---|
| PhoBERT-large | 256 | 16 | 1e-5 | ~43 phút |
| XLM-R-large | 512 | 8 | 1e-5 | ~83 phút |
| InfoXLM-large | 512 | 8 | 1e-5 | ~83 phút |

**Tổng khoảng 3,5 giờ**, trong hạn mức 30 giờ/tuần.

Con số PhoBERT là cho trường hợp cả ba seed học đủ ba epoch. Lượt 27/08 chỉ mất 32,7 phút vì
một seed dừng sớm — cơ chế dừng sớm tiết kiệm khoảng mười phút mỗi seed chết.

## Chạy một lần cả ba, hay tách ba phiên?

**Chạy một lần cả ba là được**, và đó là mặc định của notebook này.

Hạ xung vì nhiệt (đo ở T08, mục 5 `CLAUDE.md`) làm card **chậm đi** 10–15 % khi chạy liên tục,
nhưng nó **không làm sai kết quả**. Toàn bộ macro-F1, accuracy, F1 từng lớp — thứ mà mốc so
sánh này sinh ra để đo — không hề bị ảnh hưởng.

Thứ duy nhất bị ảnh hưởng là cột **`ms/mẫu`**: mô hình chạy sau trông chậm hơn thực lực. Mà cột
đó dùng cho E11 để so **nhóm phương pháp** (bộ mã hóa vs chú ý nội tại vs Gemini), không phải
so PhoBERT với XLM-R. Chênh lệch giữa ba mô hình vốn đã tới từ độ dài chuỗi — 256 với 512
token, tức khoảng gấp đôi — nên 15 % hạ xung không đổi thứ hạng hay bậc độ lớn.

**Script tự đo nhiệt độ và xung nhịp**, in cảnh báo nếu phát hiện hạ xung, và ghi số liệu vào
`results/runs.jsonl`. Nên chạy xong sẽ biết chắc có bị hay không thay vì phải phỏng đoán.

Tách ba phiên riêng chỉ đáng làm nếu sau này cần so `ms/mẫu` giữa ba mô hình với nhau một cách
chặt chẽ. Lúc đó chạy lại từng cái là được, điểm số không phải chạy lại.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: e4996e5 T18: lưu dự đoán thô của E09, bỏ InfoXLM khỏi notebook chạy lại (#45)


In [2]:
# Ô 2 — cài đặt. pyvi là phụ thuộc mới của T18, dùng để tách từ cho PhoBERT.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate pyvi

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires bitsandbytes

In [3]:
# Ô 3 — chuẩn bị dữ liệu. E09 đọc data/interim nên phải chuẩn hóa và chia tập trước.
# Chỉ cần bộ vihallu, nên dùng --only cho nhanh. Khoảng 1 phút, chạy CPU.
# Copy output dán vào PR.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, ba mô hình đều mở nên vẫn chạy được")

get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py --only vihallu")
get_ipython().system("python -m pytest tests/test_encoder.py tests/test_splits.py -q")

HF_TOKEN: không có, ba mô hình đều mở nên vẫn chạy được

MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : e4996e5 T18: lưu dự đoán thô của E09, bỏ InfoXLM khỏi notebook chạy lại (#45)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.16.1
  bitsandbytes     : chưa cài
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/uni

## Hai mô hình

Chạy hai ô liền nhau, khoảng **2 giờ 15 phút**. InfoXLM bị bỏ qua, lý do ở ô 6.

Lần chạy này để **lấy lại dự đoán thô**. Lượt 27/08 cho đúng những con số dưới đây nhưng chỉ
lưu phần đã gộp, nên chỉ số nhị phân thêm ở T19 không tính lại được mà phải chạy lại.

| | macro-F1 | ghi chú |
|---|---|---|
| PhoBERT, 3 seed | 0,7416 ± 0,0118 | cả ba seed đều học được |
| XLM-R, 3 seed | 0,7708 ± 0,0249 | cả ba seed đều học được |

**Nếu lần này ra khác đáng kể thì có gì đó sai**, vì hạt giống, cách chia tập và cấu hình đều
không đổi. Chênh lệch nhỏ ở vài chữ số cuối là bình thường: phép tính trên GPU không hoàn toàn
tất định.

Hai lưới lọc vẫn nguyên: seed nào loss đứng ở `ln(3)` hết một epoch thì dừng sớm và bị loại,
seed nào chỉ đoán một lớp cũng bị loại. Cần cả hai vì mỗi lưới có kẽ hở riêng.

In [4]:
# Ô 4 — PhoBERT-large. Khoảng 43 phút với 3 seed.
!python scripts/train_encoder_baseline.py --model phobert


T18 / E09 — TINH CHỈNH PHOBERT
  mô hình               : vinai/phobert-large
  độ dài tối đa         : 256 token
  tách từ tiếng Việt    : có
  batch / epoch / lr    : 16 / 3 / 1e-05
  số seed               : 3
  thiết bị              : cuda
  train / test          : 5,600 / 700 mẫu
config.json: 100%|█████████████████████████████| 558/558 [00:00<00:00, 2.51MB/s]
vocab.txt: 895kB [00:00, 70.1MB/s]
bpe.codes: 1.14MB [00:00, 96.7MB/s]
tokenizer.json: 3.13MB [00:00, 127MB/s]
  mẫu bị cắt vì quá dài : 29.3 %
  bộ đệm tách từ        : CacheInfo(hits=0, misses=12600, maxsize=8192, currsize=8192)
  nhiệt độ trước khi chạy : 39 °C, xung SM 300/1590 MHz (lúc rảnh, không dùng để xét hạ xung)

  --- seed 42 (1/3) ---
pytorch_model.bin: 100%|████████████████████| 1.48G/1.48G [00:09<00:00, 164MB/s]
model.safetensors:   5%|▉                   | 67.2M/1.48G [00:01<00:07, 198MB/s]
Loading weights: 100%|█████████████████████| 389/389 [00:00<00:00, 15300.50it/s]
      nạp trọng số: thân nạp đủ; đầu phân

In [5]:
# Ô 5 — XLM-R-large. Khoảng 91 phút với 3 seed.
!python scripts/train_encoder_baseline.py --model xlmr


T18 / E09 — TINH CHỈNH XLMR
  mô hình               : FacebookAI/xlm-roberta-large
  độ dài tối đa         : 512 token
  tách từ tiếng Việt    : không
  batch / epoch / lr    : 8 / 3 / 1e-05
  số seed               : 3
  thiết bị              : cuda
  train / test          : 5,600 / 700 mẫu
config.json: 100%|█████████████████████████████| 616/616 [00:00<00:00, 2.89MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 154kB/s]
sentencepiece.bpe.model: 100%|██████████████| 5.07M/5.07M [00:00<00:00, 189MB/s]
tokenizer.json: 9.10MB [00:00, 100MB/s]
  mẫu bị cắt vì quá dài : 2.5 %
  nhiệt độ trước khi chạy : 67 °C, xung SM 300/1590 MHz (lúc rảnh, không dùng để xét hạ xung)

  --- seed 42 (1/3) ---
model.safetensors: 100%|████████████████████| 2.24G/2.24G [00:10<00:00, 207MB/s]
Loading weights: 100%|██████████████████████| 389/389 [00:00<00:00, 2267.42it/s]
      nạp trọng số: thân nạp đủ; đầu phân loại mới khởi tạo 4 trọng số, bỏ qua 7 trọng số của tác vụ cũ
      D

In [6]:
# Ô 6 — InfoXLM-large: BỎ QUA. Đã chốt ở lượt 27/08 là không tinh chỉnh được, và ba lượt
# thử ở 1e-5, 5e-6, 2e-6 đều đứng ở ln(3). Chạy lại chỉ tốn khoảng 30 phút để in ra cùng
# một kết luận. Bỏ cấu hình khỏi chú thích dưới đây nếu có lý do mới để thử lại.
#
# !python scripts/train_encoder_baseline.py --model infoxlm
print("bỏ qua InfoXLM — xem phần T18 trong TASKS.md")

bỏ qua InfoXLM — xem phần T18 trong TASKS.md


In [7]:
# Ô 7 — lấy kết quả về. results/ không đẩy ngược lên GitHub từ notebook được,
# nên tải file này xuống rồi commit từ máy cá nhân.
import shutil

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
with open("results/runs.jsonl", encoding="utf-8") as handle:
    print(handle.read())

{"config": {"dataset": {"name": "vihallu", "split_seed": 42}, "encoder": {"batch_size": 16, "epochs": 3, "lr": 1e-05, "max_length": 256, "name": "vinai/phobert-large", "segment": true}, "experiment": "E09"}, "config_hash": "00176fc784cf", "extra": {"ci_from_seed": 42, "gpu": "Tesla T4", "gpu_telemetry": [{"clock_ratio": 0.9905660377358491, "sm_clock_max_mhz": 1590.0, "sm_clock_mhz": 1575.0, "temperature_c": 76.0, "utilization_pct": 95.0}, {"clock_ratio": 0.9905660377358491, "sm_clock_max_mhz": 1590.0, "sm_clock_mhz": 1575.0, "temperature_c": 76.0, "utilization_pct": 97.0}, {"clock_ratio": 0.9811320754716981, "sm_clock_max_mhz": 1590.0, "sm_clock_mhz": 1560.0, "temperature_c": 76.0, "utilization_pct": 99.0}], "gpu_throttled": false, "learned_per_seed": [true, true, true], "macro_f1_per_seed": [0.7394676900831328, 0.7542913881623559, 0.7310699717880378], "ms_per_sample": 12.232534346666549, "n_params_trainable": 369166339, "n_seeds": 3, "n_seeds_failed": 0, "note": "Bản ghi dựng lại ngày

## Đọc kết quả thế nào

Việc chính của lần chạy này là **điền hai ô nhị phân** trong Bảng 1 của `docs/EXPERIMENTS.md`.
Chúng để trống vì lượt trước không lưu dự đoán thô.

Kiểm tra trước tiên: **macro-F1 phải khớp lượt 27/08** — PhoBERT 0,7416 và XLM-R 0,7708. Lệch
nhiều nghĩa là có gì đó đã đổi ngoài ý muốn, đừng dùng số mới.

Bốn con số cần chú ý:

1. **macro-F1 trung bình qua các seed**, kèm khoảng tin cậy 95 % từ bootstrap tập test.
2. **F1 lớp `intrinsic`.** Lớp khó nhất ở mọi phương pháp: E01 đạt 0,533, Gemini 0,582,
   XLM-R 0,722. Đây là chỗ `chunk-aware` phải chứng minh mình.
3. **macro-F1 từng seed.** Seed bị đánh dấu `KHÔNG HỌC ĐƯỢC` hoặc `SỤP ĐỔ` đã bị loại khỏi
   thống kê. Kiểm tra nhanh: **trung bình phải nằm trong khoảng tin cậy** in cùng dòng — nằm
   ngoài là dấu hiệu của lỗi, không phải của nhiễu.
4. **Tỷ lệ mẫu bị cắt vì quá dài:** 29,3 % ở giới hạn 256 token của PhoBERT, 2,5 % ở 512.
   Nếu PhoBERT thua thì phải ghi rõ nó thua một phần vì **không đọc hết được ngữ cảnh**.

## Nếu hết bộ nhớ

Giảm batch trước, đừng giảm độ dài — độ dài đang là chỗ PhoBERT vốn đã thiệt. Sửa `MODELS`
trong `src/vihallulens/detect/encoder.py`.